In [ ]:
!pip install tellurium
!pip install numpy==1.24.4

In [ ]:
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the model with dosing events for one-time exposure
one_time_exposure_model = '''
model wood_smoke_one_time_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 1.0;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;


  // Wood smoke dosing event
  E1: at (1 < time < 2): WS = 1.0;
  E2: at (time >= 2): WS = 0.0;
end
'''

# Define the model with dosing events for continuous exposure
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

  // Parameters
  E_h = 1.0;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;


  // Wood smoke continuous exposure
  E1: at (time > 1): WS = 1.0;
end
'''

# Define the model with dosing events for recurring exposure
recurring_exposure_model = '''
model wood_smoke_recurring_exposure
  // Species
  species E_h, E_d, P

  // Parameters
  E_h = 1.0;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;


  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
end
'''

# Function to simulate and plot a model
def simulate_and_plot(model_code, title, label_fontsize=14, tick_fontsize=12):
    r = te.loada(model_code)
    result = r.simulate(0, 72, 1000)

    # Use a colorblind-friendly palette
    colors = sns.color_palette("colorblind")
    line_styles = ['-', '--', '-.', ':']

    plt.figure(figsize=(10, 6))
    for i, species in enumerate(['$E_h$', '$E_d$', '$P$']):
        plt.plot(result[:,0], result[:,i+1], label=species, color=colors[i], linestyle=line_styles[i])

    plt.xlabel('Time (hours)', fontsize=label_fontsize)
    plt.ylabel('Normalized Concentration', fontsize=label_fontsize)
    plt.legend(fontsize=label_fontsize)
    plt.xticks(fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    # plt.title(title, fontsize=label_fontsize)
    # plt.ylim(0, 0.25)
    plt.xlim(0, 72)

    plt.savefig(f'{title}.png', format='png')
    plt.savefig(f'{title}.eps', format='eps')
    plt.show()

# Simulate and plot one-time exposure
simulate_and_plot(one_time_exposure_model, 'Wood Smoke One-Time Exposure Simulation')

# Simulate and plot continuous exposure
simulate_and_plot(continuous_exposure_model, 'Wood Smoke Continuous Exposure Simulation')

# Simulate and plot recurring exposure (3 meals a day)
simulate_and_plot(recurring_exposure_model, 'Wood Smoke Recurring Exposure Simulation (3 meals a day)')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the updated model with continuous exposure
continuous_exposure_model = """
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;

end
"""

# Create a Tellurium model
r = te.loada(continuous_exposure_model)

# Use a colorblind-friendly palette
colors = sns.color_palette("colorblind")

# Define line styles
line_styles = ['-', '--', '-.', ':']

# Parameter scan for varying damage parameter e3
e3_values = [0.001, 0.002, 0.005, 0.01]
plt.figure(figsize=(10, 5))
for i, e3 in enumerate(e3_values):
    r.resetAll()
    r.e3 = e3
    result = r.simulate(0, 72, 1000, ['time', 'E_d'])
    plt.plot(result[:, 0], result[:, 1], label=f'$e_3={e3}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Damaged Epithelial Cells ($E_d$)', fontsize=14)
plt.legend(fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlim(0, 72)
plt.ylim(0, max(result[:, 1]))
#plt.title('Effects of Varying Damage Parameter $e_3$', fontsize=16)
plt.savefig('Effects_of_Varying_Damage_Parameter_e3.png', format='png')
plt.savefig('Effects_of_Varying_Damage_Parameter_e3.svg', format='svg')
plt.savefig('Effects_of_Varying_Damage_Parameter_e3.eps', format='eps')
plt.show()

# Parameter scan for varying repair parameter e4
e4_values = [0.001, 0.002, 0.005, 0.01]  # Various values for repair rate e4
plt.figure(figsize=(10, 5))
for i, e4 in enumerate(e4_values):
    r.resetAll()
    r.e4 = e4
    result = r.simulate(0, 72, 1000, ['time', 'E_d'])
    plt.plot(result[:, 0], result[:, 1], label=f'$e_4={e4}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Damaged Epithelial Cells ($E_d$)', fontsize=14)
plt.legend(fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlim(0, 72)
plt.ylim(0, 0.25)
#plt.title('Effects of Varying Repair Parameter $e_4$', fontsize=16)
plt.savefig('Effects_of_Varying_Repair_Parameter_e4.png', format='png')
plt.savefig('Effects_of_Varying_Repair_Parameter_e4.svg', format='svg')
plt.savefig('Effects_of_Varying_Repair_Parameter_e4.eps', format='eps')
plt.show()


In [ ]:
# Import required libraries
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt

# Define the model with dosing events for recurring exposure (3 meals a day)
recurring_exposure_model = '''
model wood_smoke_recurring_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;


  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
end
'''

# Function to simulate and plot a model
def simulate_and_plot(model_code, title):
    r = te.loada(model_code)
    result = r.simulate(0, 72, 1000)
    plt.figure(figsize=(10, 6))
    plt.plot(result[:,0], result[:,1], label='E_h')
    plt.plot(result[:,0], result[:,2], label='E_d')
    plt.plot(result[:,0], result[:,3], label='P')
    plt.xlabel('Time (hours)')
    plt.ylabel('Concentration')
    plt.title(title)
    plt.legend()
    plt.savefig(f'{title}.png', format='png')
    plt.show()

# Simulate and plot recurring exposure (3 meals a day)
simulate_and_plot(recurring_exposure_model, 'Wood Smoke Recurring Exposure Simulation (3 meals a day)')




In [ ]:
import tellurium as te

# Define the model
r = te.loada('''
    model test_model
        # Define the species
        species M0, M1

        # Define the parameters
        parameter k1, P

        # Define the reaction
        J1: M0 -> M1; k1 * F(P) * M0

        # Define the function F(P)
        function F(P)
            1 / (1 + exp(-P))
        end

        # Initial conditions
        M0 = 1.0  # Initial concentration of M0
        M1 = 0.0  # Initial concentration of M1
        k1 = 0.1  # Rate constant
        P = 0.5   # Example value of P
    end
''')

# Simulate the model
result = r.simulate(0, 50, 100)

# Plot the results
r.plot(result)